In [1]:
from torch import nn

class TestModel(nn.Module):
    def __init__(self, input_size=3, intermediate_size=5, output_size=2):
        super(TestModel, self).__init__()
        self.fc1 = nn.Linear(input_size, intermediate_size)
        self.fc2 = nn.Linear(intermediate_size, output_size)

    def forward(self, x):
        return self.fc2(self.fc1(x))

model = TestModel()

In [6]:
from torch.optim import Adam
import torch
from pytorch_optimizer import DelayedOptimizationWrapper
from delay_optimizer.delays.distributions import Stochastic, Undelayed

bias_params = [p for n, p in model.named_parameters() if "bias" in n]
non_bias_params = [p for n, p in model.named_parameters() if "bias" not in n]
optimizer = DelayedOptimizationWrapper(Adam([{"params": bias_params, "delay": Undelayed()},{"params": non_bias_params, "delay":Stochastic(3, 1000)}], lr=0.01))

In [7]:
optimizer

DelayedOptimizationWrapper (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    delay: Undelayed
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    history: [tensor([], size=(0, 5)), tensor([], size=(0, 2))]
    lr: 0.01
    max_L: 0
    maximize: False
    weight_decay: 0

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    delay: Stochastic
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    history: [tensor([[[ 5.3105e-01,  6.8636e-02, -5.2133e-01],
         [ 2.3278e-01,  5.1172e-01,  3.4246e-01],
         [-6.9846e-02,  1.7337e-05,  2.4849e-01],
         [-2.2138e-01,  1.1057e-01,  4.9824e-01],
         [-5.3136e-02,  4.3500e-01, -5.1018e-01]],

        [[ 5.3105e-01,  6.8636e-02, -5.2133e-01],
         [ 2.3278e-01,  5.1172e-01,  3.4246e-01],
         [-6.9846e-02,  1.7337e-05,  2.4849e-01],
         [-2.2138e-01,  1.1057e-01,  4.9824e-01],
         [-5.31

In [26]:
optimizer.zero_grad()
optimizer.apply_delays()

x = torch.randn(1, 3)
y = model(x)
loss = torch.square(y).sum()
loss.backward()
optimizer.step()

In [27]:
optimizer.state[bias_params[0]].get("step", torch.tensor(0.0))

tensor(15.)

In [37]:
param_history = torch.randint(0, 100, (1, 3))
param = torch.tensor([1.0, 2.0, 3.0])
L =1